# Phase 2b — Screening Eval
**FSQA** | fine-tune pipeline

> **Bring your own data.** No data ships with this repo. This notebook reads `./local-data/phase1_fsqa_eval/<run id>/questions.jsonl` (JSON Lines, one question per line, as Phase 1 writes it). With `QUESTION_RUN_ID = None` it uses the last run folder in name order. On first run it writes the A/B split to `./local-data/phase2_splits/split_<Phase 1 run id>.json` and reuses it after that. Run Phase 1 first, or supply your own file in that format. Each question record needs `id`, `question`, `reference_answer`, `domain`, `difficulty` and `seed_section`; `citation` is read if present.

Narrows the candidate configs down to a shortlist for Phase 2c, which confirms the
winner against a sealed holdout.

**Design**
- Splits the Phase 1 question set into **set A (screen)** and **set B
  (holdout)**, stratified on domain x difficulty, fixed seed. The split is
  written to the volume once and never regenerated, so Phase 2c reads the
  same partition.
- **Set B is not loaded in this notebook.** Nothing here touches it.
- Models: the Phase 2 candidates still in contention (see `CANDIDATE_MODELS`).
- Dual judge, unchanged from Phase 2 so results stay comparable.

**Changed from Phase 2**
1. Fabrication means *contradicts the reference*, not *absent from it*. It is
   recorded as a diagnostic and never modifies the score. The old rule
   lowered the rank correlation between the two judges.
2. Judge concurrency is set separately and higher, because grading is the
   slow stage.
3. Empty judge responses get retried instead of dropped.
4. Cost is computed from tokens at write time. OpenRouter reports `cost: 0`
   on calls made with your own key, so a score-per-dollar table built from
   reported cost would read zeros.

---
## Notebook setup
1. Secret `your-modal-secret` -> `OPENROUTER_API_KEY`
2. Volume `<MODAL_VOLUME_NAME>` -> `./local-data`

In [ ]:
import concurrent.futures
import json
import os
import random
import statistics
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

print("imports ok", flush=True)

## Config

In [ ]:
# --- Screen size ----------------------------------------------------------
# Set A holds about half of your Phase 1 questions. SCREEN_N is how many of them this run uses.
# Eval calls = SCREEN_N x the number of configs; a larger SCREEN_N gives a sturdier shortlist.
# The run cap RUN_BUDGET_USD (below) bounds spend. A screen that would cost more stops at the cap.
# Anything left over in A stays available; it is NOT part of the B holdout.
SCREEN_N = 150

# --- Models ---------------------------------------------------------------
CANDIDATE_MODELS = [
    "openai/gpt-5.4",
    "openai/gpt-5.5",
    "openai/gpt-5.6-luna",
    "openai/gpt-5.6-terra",
    "openai/gpt-5.6-sol",
]
CANDIDATE_EFFORTS = ["minimal", "low", "medium", "high", "xhigh"]
PRO_VARIANTS = ["openai/gpt-5.6-sol", "openai/gpt-5.6-terra", "openai/gpt-5.6-luna"]
INCLUDE_PRO = True          # 5 x 5 + 3 = 28 configs

# --- Judges (unchanged from Phase 2, deliberately) ------------------------
JUDGES = {
    "gemini": {
        "slug": "google/gemini-3.1-pro-preview",
        "provider": {"order": ["google-vertex", "google-ai-studio"], "allow_fallbacks": True},
    },
    "kimi": {
        "slug": "moonshotai/kimi-k3",
        "provider": {"order": ["digitalocean"], "allow_fallbacks": False},
    },
}
USE_JUDGES = ["gemini", "kimi"]

EVAL_PROVIDER = {"order": ["openai"], "allow_fallbacks": False}

# --- OpenRouter list prices, $/M tokens (input, output) -------------------
# Used to compute real spend and the worst-case reservation. Each figure is the highest price listed across
# the model's OpenRouter endpoints, tiers and context bands (an upper bound). Source for each model:
# https://openrouter.ai/api/v1/models/<slug>/endpoints (gpt-5.4, gpt-5.6-luna, gpt-5.6-terra: the model
# page https://openrouter.ai/<slug>), retrieved 2026-11-05.
# A model with no confirmed price must NOT be given a guessed one: leave it out and the run refuses to start.
PRICE = {
    "openai/gpt-5.4":       (2.75, 16.50),
    "openai/gpt-5.5":       (25.00, 112.50),
    "openai/gpt-5.6-luna":  (0.80, 3.60),
    "openai/gpt-5.6-terra": (2.20, 13.20),
    "openai/gpt-5.6-sol":   (8.80, 33.00),
    "google/gemini-3.1-pro-preview": (7.20, 32.40),
    "moonshotai/kimi-k3":   (4.50, 22.50),
}

# --- Budget ---------------------------------------------------------------
# Hard ceiling for the run, in USD. Every OpenRouter attempt is reserved at its worst case before
# it is sent and settled to its actual cost after. An attempt that would cross this is not sent.
# Default 25.00. Override with the environment variable RUN_BUDGET_USD (a finite number above 0, else the run refuses to start; a blank value is not unset).
import math
import os

# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD (this notebook reads no other budget variable): unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.

# --- Paths ----------------------------------------------------------------
VOLUME_MOUNT = "./local-data"
QUESTIONS_SUBDIR = "phase1_fsqa_eval"
OUTPUT_SUBDIR = "phase2b_screen"
SPLIT_DIR = "phase2_splits"
QUESTION_RUN_ID = None      # None = most recent Phase 1 run

# Spend ledger and runner lock root, shared by every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger.
# On Modal there is no git checkout, so set SPEND_LEDGER_DIR to a path on the volume. Run ledgers live in
# <root>/<RUN_ID>/spend_ledger.jsonl and the single runner lock is <root>/runner.lock.
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


SPEND_ROOT = _ledger_root()

# --- Execution ------------------------------------------------------------
EVAL_WORKERS = 16
JUDGE_WORKERS = 32          # grading is the bottleneck
REQUEST_TIMEOUT = 300
MAX_RETRIES = 3
MAX_TOKENS = 32000          # xhigh+pro can truncate at 16k
SEED = 20260824

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY missing. Attach 'your-modal-secret'.")
if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(f"{VOLUME_MOUNT} not mounted.")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-OpenRouter-Title": "labs-benchmark",
}
random.seed(SEED)

# Per-run folder and spend ledger. Every model this notebook can call (candidates and judges) must have a
# confirmed price in PRICE: the ledger cannot bound a call it cannot price, so the run refuses to start
# before any paid call rather than spend unbounded money.
_models_used = set(CANDIDATE_MODELS) | set(PRO_VARIANTS if INCLUDE_PRO else []) | {JUDGES[j]["slug"] for j in USE_JUDGES}
missing_prices = sorted(_models_used - set(PRICE))
if missing_prices:
    raise RuntimeError(f"No confirmed price for model(s) {missing_prices}. Nothing was sent. Look up the listed "
                       "OpenRouter price (USD per million tokens, input and output) and add it to PRICE in the config "
                       "cell with its source URL and date, then rerun.")
for _m, _p in PRICE.items():
    if not (len(_p) == 2 and all(isinstance(x, (int, float)) and math.isfinite(x) and x > 0 for x in _p)):
        raise RuntimeError(f"PRICE entry for {_m} is not a pair of finite positive numbers. Nothing was sent.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. An exclusive lock file at SPEND_ROOT/runner.lock: a second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died without exiting)
# is never cleared automatically: confirm no runner is alive, then delete SPEND_ROOT/runner.lock by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this run, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)

n_cfg = len(CANDIDATE_MODELS) * len(CANDIDATE_EFFORTS) + (len(PRO_VARIANTS) if INCLUDE_PRO else 0)
print(f"Models    : {len(CANDIDATE_MODELS)}")
print(f"Configs   : {n_cfg}")
print(f"Screen N  : {SCREEN_N}")
print(f"Eval calls: ~{n_cfg * SCREEN_N:,}")
print(f"Judge calls: ~{n_cfg * SCREEN_N * len(USE_JUDGES):,}")
print(f"Workers   : eval {EVAL_WORKERS}, judge {JUDGE_WORKERS}")
print(f"Run cap   : ${RUN_BUDGET_USD:.2f}   ledger: {LEDGER_PATH}")

## A/B split

Created once and cached on the volume. If the split file already exists it is
loaded, never regenerated — otherwise the holdout would reshuffle between
runs and stop being a holdout.

In [ ]:
runs = sorted(os.listdir(os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR)))
rid = QUESTION_RUN_ID or runs[-1]
qpath = os.path.join(VOLUME_MOUNT, QUESTIONS_SUBDIR, rid, "questions.jsonl")
ALL_Q = [json.loads(l) for l in open(qpath)]
print(f"Phase 1 run : {rid}")
print(f"Questions   : {len(ALL_Q)}")

split_dir = os.path.join(VOLUME_MOUNT, SPLIT_DIR)
os.makedirs(split_dir, exist_ok=True)
split_path = os.path.join(split_dir, f"split_{rid}.json")


def stratified_split(questions, seed=SEED):
    """Half to A, half to B, balanced within each (domain, difficulty) cell."""
    rng = random.Random(seed)
    cells = defaultdict(list)
    for q in questions:
        cells[(q["domain"], q.get("difficulty", "unknown"))].append(q["id"])
    a, b = [], []
    for key in sorted(cells):
        ids = sorted(cells[key])
        rng.shuffle(ids)
        half = len(ids) // 2
        a.extend(ids[:half])
        b.extend(ids[half:])
    return sorted(a), sorted(b)


if os.path.exists(split_path):
    sp = json.load(open(split_path))
    print(f"\nSplit loaded from {split_path}")
    print(f"  created {sp['created_at']}")
else:
    a_ids, b_ids = stratified_split(ALL_Q)
    sp = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "question_run_id": rid,
        "seed": SEED,
        "set_a": a_ids,
        "set_b": b_ids,
    }
    with open(split_path, "w") as f:
        json.dump(sp, f, indent=2)
    print(f"\nSplit created and written to {split_path}")

A_IDS, B_IDS = set(sp["set_a"]), set(sp["set_b"])
assert not (A_IDS & B_IDS), "Split leaked: an id is in both sets"

SET_A = [q for q in ALL_Q if q["id"] in A_IDS]
print(f"\nSet A (screen) : {len(SET_A)}")
print(f"Set B (holdout): {len(B_IDS)}  <- not loaded here")
print(f"  A by domain     : {dict(Counter(q['domain'] for q in SET_A))}")
print(f"  A by difficulty : {dict(Counter(q['difficulty'] for q in SET_A))}")

## Stratified sample from A

Proportional across (domain, difficulty), so a few sections (FSVP
definitions sections in particular) cannot dominate the sample.

In [ ]:
def stratified_sample(questions, n, seed=SEED):
    if n >= len(questions):
        return list(questions)
    rng = random.Random(seed + 1)
    cells = defaultdict(list)
    for q in questions:
        cells[(q["domain"], q.get("difficulty", "unknown"))].append(q)
    total = len(questions)
    out = []
    for key in sorted(cells):
        pool = sorted(cells[key], key=lambda x: x["id"])
        rng.shuffle(pool)
        take = max(1, round(n * len(cells[key]) / total))
        out.extend(pool[:take])
    rng.shuffle(out)
    return out[:n]


EVAL_QUESTIONS = stratified_sample(SET_A, SCREEN_N)

print(f"Sampled: {len(EVAL_QUESTIONS)}")
print(f"  by domain     : {dict(Counter(q['domain'] for q in EVAL_QUESTIONS))}")
print(f"  by difficulty : {dict(Counter(q['difficulty'] for q in EVAL_QUESTIONS))}")
print(f"  sections      : {len(set(q['seed_section'] for q in EVAL_QUESTIONS))}")
assert all(q["id"] in A_IDS for q in EVAL_QUESTIONS), "Sample escaped set A"
print("\nHoldout integrity: OK — no set B ids in this sample.")

## Request helper

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def call_openrouter(body, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES,
                    retry_on_empty=False, purpose="call"):
    """POST and normalize. Never raises for a provider error. Every attempt, retries included, goes through
    the spend ledger: reserved at its worst case before it is sent, settled after. BudgetExceeded is not
    caught here, so a refused dispatch stops the run. retry_on_empty covers judge replies that come back
    with empty content."""
    body = openrouter_body(body)            # provider.max_price and usage accounting on every request
    last = None
    for attempt in range(retries):
        aid = ledger_reserve(body, purpose)     # raises BudgetExceeded before anything is sent
        usage = None
        try:
            r = requests.post(OPENROUTER_URL, headers=HEADERS, json=body, timeout=timeout)

            if r.status_code in (429, 503):
                wait = r.headers.get("Retry-After")
                time.sleep(float(wait) if wait and wait.isdigit() else 2 ** (attempt + 2))
                last = f"HTTP {r.status_code}"
                continue

            if r.status_code >= 400:
                last = f"HTTP {r.status_code}: {r.text[:300]}"
                if 400 <= r.status_code < 500 and r.status_code != 429:
                    return {"ok": False, "error": last}
                time.sleep(2 ** attempt)
                continue

            d = r.json()
            usage = d.get("usage") or None
            if "error" in d:
                return {"ok": False, "error": str(d["error"])[:300]}

            text = (d["choices"][0]["message"].get("content") or "").strip()
            if not text and retry_on_empty and attempt < retries - 1:
                last = "empty content"
                time.sleep(1 + attempt)
                continue

            u = d.get("usage", {}) or {}
            return {
                "ok": True, "text": text,
                "model": d.get("model"), "provider": d.get("provider"),
                "is_byok": u.get("is_byok"),
                "prompt_tokens": u.get("prompt_tokens", 0),
                "completion_tokens": u.get("completion_tokens", 0),
                "reasoning_tokens": (u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0),
                "error": None,
            }
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:300]}"
            time.sleep(2 ** attempt)
        finally:
            ledger_settle(aid, body, usage)
    return {"ok": False, "error": last}


def token_cost(model, pin, pout):
    p = PRICE.get(model)
    return 0.0 if not p else pin / 1e6 * p[0] + pout / 1e6 * p[1]


print("helper loaded")


## Discovery + judge probes

In [ ]:
def probe_config(cfg):
    model, effort, mode = cfg
    body = {"model": model, "provider": EVAL_PROVIDER,
            "messages": [{"role": "user", "content": "Reply with the single word OK."}],
            "max_tokens": 2000, "reasoning": {"effort": effort}}
    if mode:
        body["reasoning"]["mode"] = mode
    res = call_openrouter(body, timeout=180, retries=2, purpose="probe")
    return {"model": model, "effort": effort, "mode": mode,
            "config_id": f"{model.split('/')[-1]}|{effort}{'|pro' if mode else ''}",
            "supported": res["ok"], "provider": res.get("provider"),
            "is_byok": res.get("is_byok"), "error": res.get("error")}


probe_list = [(m, e, None) for m in CANDIDATE_MODELS for e in CANDIDATE_EFFORTS]
if INCLUDE_PRO:
    probe_list += [(m, "xhigh", "pro") for m in PRO_VARIANTS]

print(f"Probing {len(probe_list)} configs...\n", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
    DISCOVERY = list(pool.map(probe_config, probe_list))

supported = [d for d in DISCOVERY if d["supported"]]
print(f"Supported: {len(supported)}/{len(DISCOVERY)}")
for m in CANDIDATE_MODELS:
    got = [d["effort"] + ("+pro" if d["mode"] else "") for d in supported if d["model"] == m]
    print(f"  {m:<26} {got or 'NONE'}")
for d in DISCOVERY:
    if not d["supported"]:
        print(f"  REJECTED {d['config_id']}: {str(d['error'])[:110]}")
print(f"\nis_byok: {dict(Counter(d['is_byok'] for d in supported))}")

print("\nJudges:")
for name in USE_JUDGES:
    j = JUDGES[name]
    res = call_openrouter({"model": j["slug"], "provider": j["provider"],
                           "messages": [{"role": "user", "content": "Reply with the single word OK."}],
                           "max_tokens": 2000}, timeout=180, retries=2, purpose="probe")
    print(f"  [{'OK  ' if res['ok'] else 'FAIL'}] {name:<8} {j['slug']:<34} "
          f"provider={res.get('provider')} byok={res.get('is_byok')} "
          f"{'' if res['ok'] else str(res.get('error'))[:110]}")

## Run matrix

In [ ]:
CONFIGS = [{"config_id": d["config_id"], "model": d["model"],
            "effort": d["effort"], "mode": d["mode"]} for d in supported]
TASKS = [(c, q) for c in CONFIGS for q in EVAL_QUESTIONS]

print(f"Configs    : {len(CONFIGS)}")
print(f"Questions  : {len(EVAL_QUESTIONS)}")
print(f"Eval calls : {len(TASKS):,}")
print(f"Judge calls: {len(TASKS) * len(USE_JUDGES):,}")

## Answer generation

In [ ]:
ANSWER_SYSTEM = """You are a food safety regulatory analyst. Answer the question directly and precisely.

Cite the specific CFR subsection your answer rests on. State any condition or exception that changes the answer. If the regulation does not address the question, say so rather than inferring a requirement.

Keep the answer under 200 words. No preamble."""


def run_eval_task(task):
    cfg, q = task
    body = {"model": cfg["model"], "provider": EVAL_PROVIDER,
            "messages": [{"role": "system", "content": ANSWER_SYSTEM},
                         {"role": "user", "content": q["question"]}],
            "max_tokens": MAX_TOKENS,
            "reasoning": {"effort": cfg["effort"]}}
    if cfg["mode"]:
        body["reasoning"]["mode"] = cfg["mode"]

    res = call_openrouter(body, purpose="eval")
    return {
        "config_id": cfg["config_id"], "model": cfg["model"],
        "effort": cfg["effort"], "mode": cfg["mode"],
        "question_id": q["id"], "question": q["question"],
        "reference_answer": q["reference_answer"],
        "reference_citation": q.get("citation", ""),
        "domain": q["domain"], "difficulty": q.get("difficulty"),
        "seed_section": q.get("seed_section"),
        "answer": res.get("text", ""), "ok": res["ok"], "error": res.get("error"),
        "is_byok": res.get("is_byok"), "provider": res.get("provider"),
        "prompt_tokens": res.get("prompt_tokens", 0),
        "completion_tokens": res.get("completion_tokens", 0),
        "reasoning_tokens": res.get("reasoning_tokens", 0),
        "cost_computed": token_cost(cfg["model"], res.get("prompt_tokens", 0),
                                    res.get("completion_tokens", 0)),
    }


run_id = RUN_ID

ckpt_dir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, "_checkpoint")
os.makedirs(ckpt_dir, exist_ok=True)
ckpt = os.path.join(ckpt_dir, f"answers_{rid}.jsonl")

done = set()
ANSWERS = []
if os.path.exists(ckpt):
    ANSWERS = [json.loads(l) for l in open(ckpt)]
    done = {(a["config_id"], a["question_id"]) for a in ANSWERS}
    print(f"Resuming: {len(ANSWERS):,} already done")

remaining = [(c, q) for c, q in TASKS if (c["config_id"], q["id"]) not in done]
print(f"Run {run_id} — {len(remaining):,} calls remaining, {EVAL_WORKERS} workers", flush=True)

import threading
_lock = threading.Lock()
_fh = open(ckpt, "a")
_t0 = time.time()
_n = [0]

def _tracked(t):
    r = run_eval_task(t)
    with _lock:
        _fh.write(json.dumps(r) + "\n")
        _fh.flush()
        _n[0] += 1
        if _n[0] % 100 == 0:
            el = time.time() - _t0
            print(f"  {_n[0]:,}/{len(remaining):,}  eta {(len(remaining)-_n[0])/(_n[0]/el)/60:.0f}m", flush=True)
    return r

if remaining:
    with concurrent.futures.ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        ANSWERS.extend(pool.map(_tracked, remaining))
_fh.close()

eval_elapsed = time.time() - _t0
ok_answers = [a for a in ANSWERS if a["ok"] and a["answer"].strip()]
print(f"\nDone in {eval_elapsed/60:.1f} min")
print(f"  succeeded : {len(ok_answers):,}/{len(ANSWERS):,}")
print(f"  cost      : ${sum(a['cost_computed'] for a in ANSWERS):.2f}  (computed from tokens)")
print(f"  is_byok   : {dict(Counter(a['is_byok'] for a in ANSWERS))}")
_f = [a for a in ANSWERS if not a["ok"]]
if _f:
    print(f"  failures  : {dict(Counter(x['config_id'] for x in _f))}")

## Judging

Fabrication is redefined. In Phase 2 it meant "not supported by the
reference," which flags good answers that cite adjacent provisions the
120-word excerpt omits. It now means
**contradicts** the reference, and it never modifies the score.

In [ ]:
JUDGE_SYSTEM = """You grade answers to US food safety regulation questions against a reference answer drawn from the actual CFR text.

Score these dimensions:

accuracy (0-4): Does the answer match the reference on substance?
  4 = fully correct
  3 = correct with a minor imprecision
  2 = partially correct, one material gap
  1 = mostly wrong
  0 = wrong

citation (0-3): Does it cite the correct subsection?
  3 = correct subsection
  2 = correct section, wrong or missing subsection
  1 = wrong section, right part
  0 = no citation, or a citation to something unrelated

completeness (0-3): Does it capture the conditions and exceptions in the reference?
  3 = all of them
  2 = the main one
  1 = mentions that conditions exist
  0 = none

contradiction (boolean): Does the answer state something that CONTRADICTS the reference, or assert a requirement, deadline or threshold that is plainly false?

This is NOT triggered by an answer citing related provisions the reference does not mention, or adding correct context beyond the reference. Regulatory answers routinely cite adjacent sections. Only flag a direct conflict with the reference or a plainly invented requirement.

Return JSON only:
{"accuracy": N, "citation": N, "completeness": N, "contradiction": true|false, "note": "one short sentence"}"""

JUDGE_TEMPLATE = """QUESTION:
{question}

REFERENCE ANSWER (ground truth, from CFR text):
{reference}
Reference citation: {ref_citation}

ANSWER TO GRADE:
{answer}

Grade it. JSON only."""


def judge_one(args):
    jn, ans = args
    j = JUDGES[jn]
    body = {"model": j["slug"], "provider": j["provider"],
            "messages": [{"role": "system", "content": JUDGE_SYSTEM},
                         {"role": "user", "content": JUDGE_TEMPLATE.format(
                             question=ans["question"], reference=ans["reference_answer"],
                             ref_citation=ans["reference_citation"], answer=ans["answer"][:4000])}],
            "max_tokens": 4000}
    res = call_openrouter(body, timeout=180, retry_on_empty=True, purpose="judge")
    out = {"judge": jn, "config_id": ans["config_id"], "question_id": ans["question_id"]}
    if not res["ok"] or not res["text"]:
        out.update({"score": None, "error": res.get("error") or "empty"})
        return out
    try:
        txt = res["text"].replace("```json", "").replace("```", "").strip()
        b = txt.find("{")
        if b > 0:
            txt = txt[b:]
        g = json.loads(txt)
        out.update({
            "accuracy": int(g.get("accuracy", 0)),
            "citation": int(g.get("citation", 0)),
            "completeness": int(g.get("completeness", 0)),
            "contradiction": bool(g.get("contradiction")),
            "note": g.get("note", ""),
            "score": int(g.get("accuracy", 0)) + int(g.get("citation", 0)) + int(g.get("completeness", 0)),
            "error": None,
        })
    except Exception as e:
        out.update({"score": None, "error": f"parse: {type(e).__name__} :: {res['text'][:120]}"})
    return out


judge_tasks = [(jn, a) for jn in USE_JUDGES for a in ok_answers]
print(f"Judging {len(judge_tasks):,} pairs, {JUDGE_WORKERS} workers", flush=True)

_j0 = time.time(); _jn = [0]


def _tj(t):
    r = judge_one(t)
    _jn[0] += 1
    if _jn[0] % 200 == 0:
        el = time.time() - _j0
        print(f"  {_jn[0]:,}/{len(judge_tasks):,}  {el:.0f}s  "
              f"eta {(len(judge_tasks)-_jn[0])/(_jn[0]/el)/60:.0f}m", flush=True)
    return r


with concurrent.futures.ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as pool:
    GRADES = list(pool.map(_tj, judge_tasks))

judge_elapsed = time.time() - _j0
graded = [g for g in GRADES if g.get("score") is not None]
print(f"\nDone in {judge_elapsed/60:.1f} min")
print(f"  graded : {len(graded):,}/{len(GRADES):,}")
_bad = [g for g in GRADES if g.get("score") is None]
if _bad:
    print(f"  failed : {dict(Counter(g['judge'] for g in _bad))}")
    print(f"  sample : {str(_bad[0].get('error'))[:140]}")

## Scoring

In [ ]:
def bootstrap_ci(vals, n=2000, alpha=0.05):
    if len(vals) < 2:
        return (None, None)
    rng = random.Random(SEED)
    m = [sum(vals[rng.randrange(len(vals))] for _ in range(len(vals))) / len(vals)
         for _ in range(n)]
    m.sort()
    return round(m[int(n * alpha / 2)], 3), round(m[int(n * (1 - alpha / 2))], 3)


paired = defaultdict(dict)
for g in graded:
    paired[(g["config_id"], g["question_id"])][g["judge"]] = g["score"]
both = [(v[USE_JUDGES[0]], v[USE_JUDGES[1]]) for v in paired.values() if len(v) == 2]

print("=" * 78)
if both:
    diffs = [abs(a - b) for a, b in both]
    try:
        corr = statistics.correlation([x[0] for x in both], [x[1] for x in both])
    except Exception:
        corr = None
    print("JUDGE AGREEMENT")
    print(f"  paired            : {len(both):,}")
    print(f"  exact match       : {sum(1 for d in diffs if d == 0)/len(diffs):.1%}")
    print(f"  within 1 point    : {sum(1 for d in diffs if d <= 1)/len(diffs):.1%}")
    print(f"  mean |difference| : {statistics.mean(diffs):.2f} of 10")
    print(f"  correlation       : {corr if corr is None else round(corr,3)}")
    print(f"  {USE_JUDGES[0]} mean {statistics.mean(x[0] for x in both):.2f}  |  "
          f"{USE_JUDGES[1]} mean {statistics.mean(x[1] for x in both):.2f}")
    if corr is not None and corr < 0.7:
        print("  *** Agreement is weak. Inspect before trusting the ranking. ***")

by_cfg = defaultdict(list); contra = defaultdict(int)
cost_by = defaultdict(float); cnt_by = defaultdict(int); rtok_by = defaultdict(int)
for g in graded:
    by_cfg[g["config_id"]].append(g["score"])
    if g.get("contradiction"):
        contra[g["config_id"]] += 1
for a in ANSWERS:
    cost_by[a["config_id"]] += a["cost_computed"]
    cnt_by[a["config_id"]] += 1
    rtok_by[a["config_id"]] += a["reasoning_tokens"]

rows = []
for cid, sc in by_cfg.items():
    lo, hi = bootstrap_ci(sc)
    cpq = cost_by[cid] / max(cnt_by[cid], 1)
    rows.append({"config_id": cid, "n": len(sc), "mean": round(statistics.mean(sc), 3),
                 "ci_low": lo, "ci_high": hi, "contradictions": contra[cid],
                 "cost_per_q": round(cpq, 5), "cost_15k": round(cpq * 15000, 0),
                 "reasoning_tokens": rtok_by[cid]})
rows.sort(key=lambda r: -r["mean"])

print("\n" + "=" * 78)
print("RANKING (max 10)")
print("=" * 78)
print(f"{'#':>3} {'config':<28}{'score':>7}{'95% CI':>16}{'contra':>8}{'$/15k':>9}")
for i, r in enumerate(rows, 1):
    ci = f"[{r['ci_low']}, {r['ci_high']}]"
    print(f"{i:>3} {r['config_id']:<28}{r['mean']:>7.2f}{ci:>16}"
          f"{r['contradictions']:>8}{r['cost_15k']:>9,.0f}")

if rows:
    top = rows[0]
    ov = [r for r in rows[1:] if r["ci_high"] is not None and r["ci_high"] >= top["ci_low"]]
    print(f"\nLeader: {top['config_id']} ({top['mean']:.2f})")
    print(f"CI overlaps leader: {len(ov)} of {len(rows)-1}")
    print("\nSHORTLIST CANDIDATES — statistically tied with the leader, cheapest first:")
    for r in sorted([top] + ov, key=lambda x: x["cost_15k"])[:8]:
        print(f"  {r['config_id']:<28} {r['mean']:>6.2f}   ${r['cost_15k']:>8,.0f} per 15k pairs")
    print("\nPick 5 of these for Phase 2c. Score alone should not decide it;")
    print("the winner generates 15,000 pairs, so cost is part of the call.")

## Write to volume

In [ ]:
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)


def dump(name, data):
    p = os.path.join(outdir, name)
    with open(p, "w") as f:
        for r in data:
            f.write(json.dumps(r) + "\n")
    return p


manifest = {
    "run_id": run_id, "stage": "2b_screen",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "question_run_id": rid, "split_file": split_path,
    "set_a_size": len(SET_A), "set_b_size": len(B_IDS),
    "screen_n": len(EVAL_QUESTIONS),
    "sampled_question_ids": sorted(q["id"] for q in EVAL_QUESTIONS),
    "configs": len(CONFIGS), "eval_calls": len(ANSWERS), "eval_ok": len(ok_answers),
    "judges": USE_JUDGES, "judge_calls": len(GRADES), "judge_ok": len(graded),
    "judge_agreement": {
        "paired": len(both),
        "correlation": round(corr, 4) if both and corr is not None else None,
        "exact_pct": round(sum(1 for d in diffs if d == 0)/len(diffs), 4) if both else None,
    },
    "eval_cost_usd": round(sum(a["cost_computed"] for a in ANSWERS), 2),
    "eval_is_byok": {str(k): v for k, v in Counter(a["is_byok"] for a in ANSWERS).items()},
    "elapsed_min": {"eval": round(eval_elapsed/60, 1), "judge": round(judge_elapsed/60, 1)},
    "ranking": rows,
    "files": {"answers": dump("answers.jsonl", ANSWERS),
              "grades": dump("grades.jsonl", GRADES),
              "discovery": dump("discovery.jsonl", DISCOVERY),
              "results": dump("results.jsonl", rows)},
}
with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Eval cost : ${manifest['eval_cost_usd']:.2f}")
print(f"Wall time : {manifest['elapsed_min']['eval']:.0f}m eval + "
      f"{manifest['elapsed_min']['judge']:.0f}m judge")
print(ledger_summary())
print(f"Output    : {outdir}")
print(f"\nSet B ({len(B_IDS)} questions) untouched. Phase 2c reads it from {split_path}")

## Inspect

Read a few gradings before choosing the shortlist. A judge rewarding
verbosity, or penalizing a correct answer worded differently, shows up here
and nowhere else.

In [ ]:
def show(config_id=None, n=3, lowest=False):
    akey = {(a["config_id"], a["question_id"]): a for a in ok_answers}
    pool_ = [g for g in graded if config_id is None or g["config_id"] == config_id]
    if lowest:
        pool_ = sorted(pool_, key=lambda g: g["score"])
    for g in pool_[:n]:
        a = akey.get((g["config_id"], g["question_id"]))
        if not a:
            continue
        print("=" * 78)
        print(f"{g['config_id']} | {g['question_id']} | {g['judge']} | {g['score']}/10")
        print(f"  acc={g.get('accuracy')} cite={g.get('citation')} "
              f"comp={g.get('completeness')} contra={g.get('contradiction')}")
        print(f"  note: {g.get('note')}")
        print(f"\nQ: {a['question'][:280]}")
        print(f"\nREF: {a['reference_answer'][:350]}")
        print(f"\nGOT: {a['answer'][:350]}\n")


if graded:
    print("Lowest-scoring gradings — check for judge error:\n")
    show(n=3, lowest=True)